# GrowthLab — Sentiment model tuning + second-model experiment
Run on Colab T4. **Model A:** tune DistilBERT on Yelp. **Model B:** optionally fine-tune the winning checkpoint on Amazon reviews, then compare on the same Yelp test and Amazon test. Domain shift is expected; no performance claims until measured. Yelp test is only used for final comparison, not hyperparameter selection.

In [1]:
!pip -q install "transformers>=4.45,<5" "datasets>=3,<5" "accelerate>=1,<2" "scikit-learn>=1.3,<2" "safetensors>=0.4" pandas matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 99.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import os, random, gc, json, shutil, numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
BASE='distilbert-base-uncased'; LABELS={i:f'{i+1}_star' for i in range(5)}
tokenizer=AutoTokenizer.from_pretrained(BASE); collator=DataCollatorWithPadding(tokenizer)
def tokenize(batch):return tokenizer(batch['text'],truncation=True,max_length=256)
def metrics(p):
    y=np.asarray(p.label_ids); pred=np.asarray(p.predictions).argmax(-1)
    return {'accuracy':accuracy_score(y,pred),'macro_f1':f1_score(y,pred,average='macro')}
def model_from(path):return AutoModelForSequenceClassification.from_pretrained(path,num_labels=5,id2label=LABELS,label2id={v:k for k,v in LABELS.items()})
def tok(ds):return ds.map(tokenize,batched=True,remove_columns=['text'])

GPU: Tesla T4


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## Dataset: keep the test split untouched until final evaluation. Increase TRAIN_N to 30000 after a successful smoke test. Validation is stratified.

In [3]:
raw=load_dataset('Yelp/yelp_review_full')
TRAIN_N=10000; VAL_N=1000; TEST_N=2000
sample=raw['train'].shuffle(seed=SEED).select(range(TRAIN_N+VAL_N))
split=sample.train_test_split(test_size=VAL_N,seed=SEED,stratify_by_column='label')
yelp_train,yelp_val=tok(split['train']),tok(split['test'])
yelp_test=tok(raw['test'].shuffle(seed=SEED).select(range(TEST_N)))
print('Yelp sizes',len(yelp_train),len(yelp_val),len(yelp_test))

README.md: 0.00B [00:00, ?B/s]

yelp_review_full/train-00000-of-00001.pa(…):   0%|          | 0.00/299M [00:00<?, ?B/s]

yelp_review_full/test-00000-of-00001.par(…):   0%|          | 0.00/23.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/650000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Yelp sizes 10000 1000 2000


## Small hyperparameter search
Three controlled trials. Choose using **Yelp validation macro-F1**, never the test set. Each trial starts from the same pretrained base. Run time is approximately several minutes on a T4.

In [4]:
TRIALS=[
 {'name':'baseline_plus_epoch','lr':2e-5,'epochs':3,'weight_decay':0.01},
 {'name':'lower_lr','lr':1e-5,'epochs':3,'weight_decay':0.01},
 {'name':'higher_lr','lr':3e-5,'epochs':3,'weight_decay':0.05},
]
trial_rows=[]
for cfg in TRIALS:
    name=cfg['name']; print('TRAINING',name)
    m=model_from(BASE)
    args=TrainingArguments(output_dir=f'./checkpoints_{name}',learning_rate=cfg['lr'],
       per_device_train_batch_size=16,per_device_eval_batch_size=32,
       num_train_epochs=cfg['epochs'],weight_decay=cfg['weight_decay'],
       warmup_ratio=0.06,lr_scheduler_type='linear',
       eval_strategy='epoch',save_strategy='epoch',load_best_model_at_end=True,
       metric_for_best_model='macro_f1',greater_is_better=True,
       fp16=torch.cuda.is_available(),report_to='none',save_total_limit=1,seed=SEED)
    tr=Trainer(model=m,args=args,train_dataset=yelp_train,eval_dataset=yelp_val,
       processing_class=tokenizer,data_collator=collator,compute_metrics=metrics)
    tr.train()
    v=tr.evaluate(yelp_val)
    out=f'./trained_{name}';tr.save_model(out);tokenizer.save_pretrained(out)
    trial_rows.append({**cfg,'val_accuracy':v['eval_accuracy'],'val_macro_f1':v['eval_macro_f1'],'path':out})
    del tr,m;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
results_df=pd.DataFrame(trial_rows).sort_values('val_macro_f1',ascending=False)
display(results_df)
BEST=results_df.iloc[0]['path']; print('BEST',BEST)

TRAINING baseline_plus_epoch


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.158200,0.995729,0.563000,0.550341
2,0.872200,1.018128,0.570000,0.560793
3,0.762700,1.048399,0.566000,0.561942


TRAINING lower_lr


...

TRAINING higher_lr


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.143100,0.997724,0.547000,0.533338
2,0.850000,1.051935,0.569000,0.558609
3,0.708100,1.096322,0.564000,0.561548


,name,lr,epochs,weight_decay,val_accuracy,val_macro_f1,path
1,lower_lr,0.00001,3,0.01,0.574,0.569386,./trained_lower_lr
0,baseline_plus_epoch,0.00002,3,0.01,0.566,0.561942,./trained_baseline_plus_epoch
2,higher_lr,0.00003,3,0.05,0.564,0.561548,./trained_higher_lr


BEST ./trained_lower_lr


## Evaluate selected Yelp model (once)

In [5]:
def evaluate_model(path,dataset,name):
    m=model_from(path)
    args=TrainingArguments(output_dir=f'./eval_{name}',per_device_eval_batch_size=32,
         report_to='none',fp16=torch.cuda.is_available())
    tr=Trainer(model=m,args=args,data_collator=collator,processing_class=tokenizer,compute_metrics=metrics)
    p=tr.predict(dataset); pred=p.predictions.argmax(-1)
    print(name,p.metrics)
    print(classification_report(p.label_ids,pred,target_names=list(LABELS.values()),digits=3))
    print(confusion_matrix(p.label_ids,pred))
    row={'model':name,'accuracy':accuracy_score(p.label_ids,pred),'macro_f1':f1_score(p.label_ids,pred,average='macro')}
    del tr,m;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return row
scores=[evaluate_model(BEST,yelp_test,'yelp_tuned_on_yelp')]

yelp_tuned_on_yelp {'test_loss': 0.918433427810669, 'test_model_preparation_time': 0.0027, 'test_accuracy': 0.6065, 'test_macro_f1': 0.6051584030108101, 'test_runtime': 3.9016, 'test_samples_per_second': 512.607, 'test_steps_per_second': 16.147}
              precision    recall  f1-score   support

      1_star      0.736     0.731     0.733       431
      2_star      0.548     0.536     0.542       425
      3_star      0.542     0.538     0.540       372
      4_star      0.527     0.496     0.511       411
      5_star      0.665     0.737     0.699       361

    accuracy                          0.607      2000
   macro avg      0.604     0.608     0.605      2000
weighted avg      0.604     0.607     0.605      2000

[[315  98  11   5   2]
 [100 228  76  18   3]
 [  5  76 200  81  10]
 [  5  11  72 204 119]
 [  3   3  10  79 266]]


## Optional second model: Amazon domain adaptation
This is a **second checkpoint** trained from the best Yelp model on product reviews, not a second campaign-performance predictor. Amazon schema differs across dataset versions; this cell inspects and maps it. If the dataset cannot load, skip this experiment without blocking the Yelp model.

In [6]:
# Amazon 5-star review data (optional). The SetFit dataset uses text and label in typical releases.
try:
    amazon_raw=load_dataset('SetFit/amazon_reviews_multi_en')
    print(amazon_raw)
    print('Columns:',amazon_raw[list(amazon_raw.keys())[0]].column_names)
except Exception as exc:
    amazon_raw=None
    print('Amazon dataset unavailable; skip second-model cells:',repr(exc))

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/47.4M [00:00<?, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['id', 'text', 'label', 'label_text'],
        num_rows: 5000
    })
})
Columns: ['id', 'text', 'label', 'label_text']


In [7]:
if amazon_raw is not None:
    key='train' if 'train' in amazon_raw else list(amazon_raw.keys())[0]
    ds=amazon_raw[key]
    print('Example:',{k:str(v)[:100] for k,v in ds[0].items()})
    # SetFit amazon_reviews_multi_en normally has text + label (0..4).
    text_col=next((x for x in ['text','review_body','review'] if x in ds.column_names),None)
    label_col=next((x for x in ['label','stars','star_rating'] if x in ds.column_names),None)
    if text_col is None or label_col is None:raise ValueError('Inspect Amazon schema and set text_col/label_col explicitly')
    df=ds.to_pandas()[[text_col,label_col]].rename(columns={text_col:'text',label_col:'label'})
    df['label']=pd.to_numeric(df['label'],errors='coerce')
    df=df.dropna(subset=['text','label']);df['label']=df['label'].astype(int)
    if set(df['label'].unique()).issubset(set(range(1,6))):df['label']-=1
    assert set(df['label'].unique()).issubset(set(range(5))), 'Unexpected label mapping; inspect manually'
    df=df[df['text'].astype(str).str.strip().str.len()>10].drop_duplicates('text')
    from sklearn.model_selection import train_test_split
    # Fixed stratified 80/10/10 subset, all disjoint; do not train on Amazon test.
    sampled=df.groupby('label',group_keys=False).sample(n=min(1200,df['label'].value_counts().min()),random_state=SEED)
    train_df,temp_df=train_test_split(sampled,test_size=0.2,random_state=SEED,stratify=sampled.label)
    val_df,test_df=train_test_split(temp_df,test_size=0.5,random_state=SEED,stratify=temp_df.label)
    amazon_train=tok(Dataset.from_pandas(train_df[['text','label']],preserve_index=False))
    amazon_val=tok(Dataset.from_pandas(val_df[['text','label']],preserve_index=False))
    amazon_test=tok(Dataset.from_pandas(test_df[['text','label']],preserve_index=False))
    print('Amazon splits:',len(amazon_train),len(amazon_val),len(amazon_test))

Example: {'id': 'en_0964290', 'text': 'Arrived broken. Manufacturer defect. Two of the legs of the base were not completely formed, so ther', 'label': '0', 'label_text': '0'}


Map:   0%|          | 0/4800 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

Amazon splits: 4800 600 600


In [8]:
if amazon_raw is not None:
    # Yelp checkpoint -> Amazon fine-tuning. Risk: forgetting Yelp; compare both test domains.
    second=model_from(BEST)
    args2=TrainingArguments(output_dir='./amazon_adapt_checkpoints',learning_rate=1e-5,
      per_device_train_batch_size=16,per_device_eval_batch_size=32,num_train_epochs=2,
      weight_decay=0.01,eval_strategy='epoch',save_strategy='epoch',
      load_best_model_at_end=True,metric_for_best_model='macro_f1',greater_is_better=True,
      fp16=torch.cuda.is_available(),report_to='none',save_total_limit=1,seed=SEED)
    tr2=Trainer(model=second,args=args2,train_dataset=amazon_train,eval_dataset=amazon_val,
       processing_class=tokenizer,data_collator=collator,compute_metrics=metrics)
    tr2.train(); SECOND='./growthlab-sentiment-amazon-adapted'
    tr2.save_model(SECOND);tokenizer.save_pretrained(SECOND)
    del tr2,second;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    scores.extend([
      evaluate_model(BEST,amazon_test,'yelp_tuned_on_amazon'),
      evaluate_model(SECOND,amazon_test,'amazon_adapted_on_amazon'),
      evaluate_model(SECOND,yelp_test,'amazon_adapted_on_yelp')])
    display(pd.DataFrame(scores))

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,1.070395,0.523333,0.508359
2,1.063500,1.057281,0.516667,0.510870


yelp_tuned_on_amazon {'test_loss': 1.1945133209228516, 'test_model_preparation_time': 0.0017, 'test_accuracy': 0.49, 'test_macro_f1': 0.4808224962796868, 'test_runtime': 0.778, 'test_samples_per_second': 771.168, 'test_steps_per_second': 24.42}
              precision    recall  f1-score   support

      1_star      0.556     0.742     0.636       120
      2_star      0.430     0.433     0.432       120
      3_star      0.358     0.408     0.381       120
      4_star      0.397     0.242     0.301       120
      5_star      0.688     0.625     0.655       120

    accuracy                          0.490       600
   macro avg      0.486     0.490     0.481       600
weighted avg      0.486     0.490     0.481       600

[[89 24  6  1  0]
 [43 52 22  2  1]
 [18 32 49 18  3]
 [ 5 11 45 29 30]
 [ 5  2 15 23 75]]


amazon_adapted_on_amazon {'test_loss': 1.0869468450546265, 'test_model_preparation_time': 0.0017, 'test_accuracy': 0.5233333333333333, 'test_macro_f1': 0.519002220030984, 'test_runtime': 0.7237, 'test_samples_per_second': 829.116, 'test_steps_per_second': 26.255}
              precision    recall  f1-score   support

      1_star      0.583     0.700     0.636       120
      2_star      0.439     0.392     0.414       120
      3_star      0.441     0.408     0.424       120
      4_star      0.441     0.433     0.437       120
      5_star      0.683     0.683     0.683       120

    accuracy                          0.523       600
   macro avg      0.518     0.523     0.519       600
weighted avg      0.518     0.523     0.519       600

[[84 29  5  2  0]
 [40 47 24  9  0]
 [13 23 49 30  5]
 [ 3  5 27 52 33]
 [ 4  3  6 25 82]]


amazon_adapted_on_yelp {'test_loss': 0.9820842742919922, 'test_model_preparation_time': 0.0017, 'test_accuracy': 0.576, 'test_macro_f1': 0.5739928490628584, 'test_runtime': 3.7699, 'test_samples_per_second': 530.518, 'test_steps_per_second': 16.711}
              precision    recall  f1-score   support

      1_star      0.767     0.694     0.728       431
      2_star      0.553     0.468     0.507       425
      3_star      0.485     0.438     0.460       372
      4_star      0.452     0.516     0.482       411
      5_star      0.627     0.773     0.692       361

    accuracy                          0.576      2000
   macro avg      0.577     0.578     0.574      2000
weighted avg      0.579     0.576     0.574      2000

[[299 105  19   6   2]
 [ 78 199 112  35   1]
 [  4  45 163 146  14]
 [  4   8  38 212 149]
 [  5   3   4  70 279]]


,model,accuracy,macro_f1
0,yelp_tuned_on_yelp,0.606500,0.605158
1,yelp_tuned_on_amazon,0.490000,0.480822
2,amazon_adapted_on_amazon,0.523333,0.519002
3,amazon_adapted_on_yelp,0.576000,0.573993


## Export both models
Choose the checkpoint that best matches your target SMEs. A domain-adapted model may improve retail reviews and hurt restaurant reviews. Test on real Malaysian examples before making deployment claims.

In [9]:
from google.colab import files
shutil.copytree(BEST,'./growthlab-sentiment-yelp-tuned',dirs_exist_ok=True)
for path in ['./growthlab-sentiment-yelp-tuned']+(['./growthlab-sentiment-amazon-adapted'] if amazon_raw is not None else []):
    archive=shutil.make_archive(path,'zip',path)
    print(archive,os.path.getsize(archive))
    files.download(archive)
pd.DataFrame(scores).to_csv('growthlab_model_comparison.csv',index=False)
files.download('growthlab_model_comparison.csv')

/content/growthlab-sentiment-yelp-tuned.zip 247320336


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/content/growthlab-sentiment-amazon-adapted.zip 247321474


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>